In [11]:
import pandas as pd

# Load data
csf_nodes = pd.read_csv("./9. PUC/csf_nodes.csv") # columns: node, direction
ctx_nodes = pd.read_csv("./9. PUC/network_nodes.csv") # columns: ID, Mean Log2 Fold Change Direction (DSS)
edges = pd.read_csv("8. Filter FDR/final_edges_with_fdr_filtered.csv") # columns: Gene, Metabolite, Sign

# Build lookup maps
# Gene -> direction, from network_nodes
gene_direction_map = ctx_nodes.set_index("ID")["Mean Log2 Fold Change Direction (DSS)"].to_dict()

# Metabolite -> direction, from csf_nodes
metabolite_direction_map = csf_nodes.set_index("node")["direction"].to_dict()

# Map values onto edges
edges["ctx_direction"] = edges["Gene"].map(gene_direction_map)
edges["cpx_direction"] = edges["Metabolite"].map(metabolite_direction_map)

# Reorder so the new columns sit right next to their source columns
cols = list(edges.columns)
cols.remove("ctx_direction")
cols.remove("cpx_direction")

gene_idx = cols.index("Gene")
cols.insert(gene_idx + 1, "ctx_direction")

# recompute Metabolite's index since the list shifted after inserting ctx_direction
metabolite_idx = cols.index("Metabolite")
cols.insert(metabolite_idx + 1, "cpx_direction")

edges = edges[cols]

# Sanity check: report any genes/metabolites that didn't find a match
missing_genes = edges.loc[edges["ctx_direction"].isna(), "Gene"].unique()
missing_metabolites = edges.loc[edges["cpx_direction"].isna(), "Metabolite"].unique()
#if len(missing_genes) > 0:
#    print(f"Warning: {len(missing_genes)} gene(s) not found in network_nodes: {missing_genes}")
#if len(missing_metabolites) > 0:
#    print(f"Warning: {len(missing_metabolites)} metabolite(s) not found in csf_nodes: {missing_metabolites}")

# Step 5 ----------------------------------
edges["predicted_effect_sign"] = edges["ctx_direction"] * edges["cpx_direction"]

# Step 6 ----------------------------------
rep_cols = ["VECPAC r", "LPS r", "DSS r", "Pooled r"]
def check_consistency(row):
    vals = row[rep_cols].dropna().unique()
    return len(vals) <= 1   # True if all same or all NaN
edges["check_consistency"] = edges.apply(check_consistency, axis=1)

# Step 7 ----------------------------------
edges["pooled_sign_agree"] = edges["Pooled r"].fillna(edges["Sign"]).eq(edges["Sign"].map({"+": 1.0, "-": -1.0}))

edges["Sign"] = edges["Sign"].map({"+": 1.0, "-": -1.0})

# Step 8 ----------------------------------
edges["within-model correlation signs agree with the predicted correlation"] = edges["Sign"].eq(edges["predicted_effect_sign"])

# Step 9 ----------------------------------
edges["pooled correlation signs agree with the predicted correlation"] = edges["Pooled r"].eq(edges["predicted_effect_sign"])

# Step 10 ----------------------------------
edges["all_signs_agree"] = edges["Sign"].eq(edges["Pooled r"]).eq(edges["predicted_effect_sign"])

#edges = edges[["Gene", "ctx_direction", "Metabolite", "cpx_direction", "Sign", "edge_direction"]]
edges = edges[[
    "Gene",
    "ctx_direction",
    "Metabolite",
    "cpx_direction",
    "VECPAC r",
    "LPS r",
    "DSS r",
    "Pooled r",
    "predicted_effect_sign",
    "check_consistency",
    "pooled_sign_agree",
    "within-model correlation signs agree with the predicted correlation",
    "pooled correlation signs agree with the predicted correlation",
    "all_signs_agree"
]]
output_path = "9. PUC/final_edges_with_fdr_filtered_with_directions.csv"

edges.to_csv(output_path, index=False)
#print(f"Saved: {output_path}")
#print(edges.head())
print(edges.columns.tolist())

['Gene', 'ctx_direction', 'Metabolite', 'cpx_direction', 'VECPAC r', 'LPS r', 'DSS r', 'Pooled r', 'predicted_effect_sign', 'check_consistency', 'pooled_sign_agree', 'within-model correlation signs agree with the predicted correlation', 'pooled correlation signs agree with the predicted correlation', 'all_signs_agree']
